# RNN + Bahdanau Attention for Text Summarization (Toy Example)

A minimal PyTorch seq2seq model with **Bahdanau (additive) attention**, trained on a synthetic dataset.

**Task:** given a random "sentence" (sequence of word ids), predict the **3 most frequent words** in it, in frequency order. This is a tiny stand-in for extractive summarization — the important words are scattered through the source, so attention actually has something to align to.

## 1. Imports & setup

In [1]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)
random.seed(0)

## 2. Synthetic dataset

Each source is 10 random word ids. The target is `[SOS, top-3 most frequent words, EOS]`.

In [2]:
VOCAB_SIZE = 20                 # word ids 3..VOCAB_SIZE-1 are "real" words
PAD, SOS, EOS = 0, 1, 2
SRC_LEN = 10
SUMMARY_LEN = 3

def make_example():
    src = [random.randint(3, VOCAB_SIZE - 1) for _ in range(SRC_LEN)]
    counts = {}
    for w in src:
        counts[w] = counts.get(w, 0) + 1
    ranked = sorted(counts, key=lambda w: (-counts[w], src.index(w)))
    tgt = ranked[:SUMMARY_LEN]
    return src, [SOS] + tgt + [EOS]

def make_batch(bs):
    srcs, tgts = zip(*[make_example() for _ in range(bs)])
    return torch.tensor(srcs), torch.tensor(tgts)

make_example()

([15, 16, 4, 11, 19, 18, 15, 12, 18, 14], [1, 15, 18, 16, 2])

## 3. Encoder

Embedding + single-layer GRU, returns all hidden states for attention to look over.

In [3]:
HID = 64

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, HID, padding_idx=PAD)
        self.gru = nn.GRU(HID, HID, batch_first=True)

    def forward(self, src):
        out, h = self.gru(self.emb(src))
        return out, h                      # out: (B,T,H)  h: (1,B,H)

## 4. Bahdanau (additive) attention

$$\text{score}(s_{t-1}, h_i) = v^\top \tanh(W_1 s_{t-1} + W_2 h_i)$$

In [4]:
class BahdanauAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.Wa = nn.Linear(HID, HID)
        self.Ua = nn.Linear(HID, HID)
        self.v = nn.Linear(HID, 1)

    def forward(self, dec_h, enc_out):
        # dec_h: (B,H)   enc_out: (B,T,H)
        score = self.v(torch.tanh(self.Wa(dec_h.unsqueeze(1)) + self.Ua(enc_out)))
        weights = torch.softmax(score, dim=1)             # (B,T,1)
        context = (weights * enc_out).sum(dim=1)           # (B,H)
        return context, weights.squeeze(-1)

## 5. Decoder

At each step: embed the input token, attend over encoder outputs, feed `[embedding ; context]` into a `GRUCell`.

In [5]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, HID, padding_idx=PAD)
        self.attn = BahdanauAttention()
        self.gru = nn.GRUCell(HID * 2, HID)
        self.out = nn.Linear(HID, VOCAB_SIZE)

    def step(self, tok, h, enc_out):
        x = self.emb(tok)                                  # (B,H)
        context, attw = self.attn(h, enc_out)               # (B,H)
        h = self.gru(torch.cat([x, context], dim=-1), h)
        return self.out(h), h, attw

## 6. Seq2Seq wrapper

Runs the decoder step-by-step with teacher forcing.

In [6]:
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.dec = Decoder()

    def forward(self, src, tgt, tf_ratio=0.5):
        enc_out, h = self.enc(src)
        h = h.squeeze(0)                                    # (B,H)
        B, T = tgt.shape
        logits = torch.zeros(B, T - 1, VOCAB_SIZE)
        tok = tgt[:, 0]                                      # SOS
        for t in range(1, T):
            step_logits, h, _ = self.dec.step(tok, h, enc_out)
            logits[:, t - 1] = step_logits
            tok = tgt[:, t] if random.random() < tf_ratio else step_logits.argmax(-1)
        return logits

## 7. Training

In [7]:
model = Seq2Seq()
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

for epoch in range(300):
    src, tgt = make_batch(64)
    logits = model(src, tgt, tf_ratio=0.5)
    loss = loss_fn(logits.reshape(-1, VOCAB_SIZE), tgt[:, 1:].reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.3f}")

epoch   0  loss 2.980


epoch  50  loss 2.265


epoch 100  loss 1.915


epoch 150  loss 1.685


epoch 200  loss 1.367


epoch 250  loss 1.447


## 8. Inference demo

Greedy decoding from `SOS` until `EOS`.

In [8]:
@torch.no_grad()
def summarize(src_seq):
    src = torch.tensor([src_seq])
    enc_out, h = model.enc(src)
    h = h.squeeze(0)
    tok = torch.tensor([SOS])
    result = []
    for _ in range(SUMMARY_LEN + 1):
        logits, h, _ = model.dec.step(tok, h, enc_out)
        tok = logits.argmax(-1)
        if tok.item() == EOS:
            break
        result.append(tok.item())
    return result

src, tgt = make_example()
print("Source      :", src)
print("True summary:", tgt[1:-1])
print("Pred summary:", summarize(src))

Source      : [7, 9, 8, 6, 16, 5, 15, 9, 12, 17]
True summary: [9, 7, 8]
Pred summary: [9, 7, 8]
